# Applications of Data Science - Summer Term 2026
- Group: 11 - International Taxation
- Group Members:
    - Simon Andreas **ERTL**
    - Berkay **KAYA**
    - Joel **PUTHENPARAMBIL**
    - Fedor **SAMOROKOV**
- Author of this notebook: Simon Andreas **ERTL**
## Group Project: Building an Austrian Tax Law Assistant
This project is split into four parts:
- Creation of a shared corpus regarding Austrian Tax Law from which each group can use for their LLM models. This corpus consists of questions regarding Austrian Tax Law focusing on different topics (e. g. International Taxation), detailed answers to these questions, and sources referenced.
- Applying models to the created corpus. This task itself is split into three parts:
    - Prompt LLMs (Inference): Out-of-the-box Large Language Models should be used to answer the questions created during Task 1.
    - Fine-tune models (Training): The models should then be fine-tuned using additional materials.
    - RAG: Using RAG, the performnance of the models should again be improved.
- Evalution of model performance
- Presentation

This notebook was created in Google Colab using the available T4 GPU architecture. To replicate the results, upload this notebook to Google Colab, and select the right architecture in the top right corner.

### Task 2
Before the tasks itself can be performed, various steps have to be completed beforehand:
- Importing necessary libaries
- Path handling
- Set up of the environment

In the following cell, the filename of the dataset (input and output) can be specified. Only the file name is needed, as the filepath will be handled seperately.

In [1]:
# Only file names are specified, the files have to be in the correct folders or the paths have to be adjusted later on
input_dataset = "dataset_clean.csv" # Defining input dataset file name
sft_dataset = "sft_dataset.json" # Defining dataset name for fine-tuning
output_dataset = "output_model2_ERTL.csv" # Defining output dataset file name

In the following code cells, the necessary libraries will be imported.

In [2]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-sfxosvl9/unsloth_fc2b9c0572df40eab60a5508558d6d0b
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-sfxosvl9/unsloth_fc2b9c0572df40eab60a5508558d6d0b
  Resolved https://github.com/unslothai/unsloth.git to commit f801e59c29db7b4028297ef987af6bfdaa464500
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 130.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.9 MB/s eta 0:00:0

In [3]:
!pip install -U trl # Altough the previous cell installs trl, the version installed did not work with the SFT modules

import pandas as pd # For dataframe handling
from pathlib import Path # For path handling
import torch # For back-end
from unsloth import FastLanguageModel # To interact with the models
import json # For JSON handling
from datasets import Dataset # Converting into HF datasets
from trl import SFTConfig, SFTTrainer # For supervised fine-tuning
from transformers import EarlyStoppingCallback # For early stopping
from google.colab import files # For downloading the results
import shutil # To zip adapter folder and download it
from time import sleep # For pause before downloading

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.5 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully uninstalled trl-0.8.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.4.4 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.8.4 which is incompatible.
unsloth-zoo 2026.4.4 requires trl!=0.19.0,<=0.24.0,>=0.18.2, but you have trl 1.0.0 which is incompatible.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broke

The cell below handles the filepath of the input and output files.

In [4]:
# Assumes code and inputs are in same directory
PROJECT_CWD = Path.cwd() # Reading the current working directory
DATASET_DIR = PROJECT_CWD / input_dataset # Setting path for the input dataset
OUTPUT_DIR = PROJECT_CWD / output_dataset # Setting path for the output dataset
SFT_DIR = PROJECT_CWD / sft_dataset # Setting path for SFT dataset
CHECKPOINT_DIR = PROJECT_CWD / "model2_checkpoints" # Path for training checkpoints
ADAPTER_OUTPUT_DIR = PROJECT_CWD / "model2_lora_adapter" # Path for adapter output

### Task 2.2.1: Fine-tuning the model and Answering prompts from Task 1
In Task 1, each group had to formulate questions and answers (with sources) regarding the Austrian Tax Law, with each group having a different focus. Our grouped focused on international taxation.
In this task, Large Language Models should be implemented so they can answer the questions produced in Task 1. Task 2b requests us to fine-tune the model used in Task 2a using additional data. For this, we have generated a JSON-file with 300 entries based on the following structure:
- ID
- Prompt
- Answer
- Sources

This dataset spans accross all topics from the Google Sheet. To generate this entries, we have exported relevant law text from RIS and used this as ChatGPT references.

First, we want all configurations centralized so they can be easily modified.

In [5]:
# Model settings
BASE_MODEL_NAME = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit" # Model source
MAX_SEQ_LENGTH = 2048 # Context length
LOAD_IN_4_BIT = True # Set bit mode, 4 bit for less memory usage

# LoRA settings
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

# Training settings
TRAIN_BATCH_SIZE = 1 # Set batch size for training
GRADIENT_ACCUMULATION_STEPS = 8 # Set gradient accumulation
NUM_EPOCHS = 1 # Set number of times training set is iterated through
LEARNING_RATE = 2e-4 # Common LoRA learning rate
LOGGING_STEPS = 5 # Log every 5 steps
SAVE_STRATEGY = "epoch" # Save once per epoch
EVAL_STRATEGY = "epoch" # Evaluate once per epoch

# Settings for early stopping; not really needed when NUM_EPOCHS = 1
EARLY_STOPPING_PATIENCE = 2 # Set patience for early stopping (if validation loss stops improving after 2 epochs)
EARLY_STOPPING_THRESHOLD = 0.0 # Treshold for early stopping
VALIDATION_SPLIT = 0.1 # Train-Validation-Split
SEED = 30112003 # Seed for reproceability

# Inference settings
MAX_NEW_TOKENS = 256 # Number of new tokens
INFER_BATCH_SIZE = 8 # Questions answered per batch

# Reproduceability
## Setting seed (depends on CUDA availability, which is available in Google Colab)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Set BF16/FP16 based on CUDA availability
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

USE_FP16 = torch.cuda.is_available() and not USE_BF16

Now we will begin handling the JSON-file which will be used for fine-tuning. For this, we will first define a helper function that will handle the sources and then a helper function which will convert the JSON-file itself.

In [6]:
def normalize_sources(value):
  # If no sources are given, an empty list will be returned
  if value is None:
    return []

  # If sources are a list, convert every entry to a list, strip it and drop empty ones
  if isinstance(value, list):
    return [str(item).strip() for item in value if str(item).strip()]

  if isinstance(value, str):
    value = value.strip()
    return [value] if value else []

  # For any other type, convert to a string and keep if not-empty
  value = str(value).strip()
  return [value] if value else []

In [7]:
def build_assistant_text(correct_answer, sources):
    assistant_text = str(correct_answer).strip() # Clean correct answer
    sources = normalize_sources(sources) # Apply source-normalization

    # If sources exist
    if sources:
        sources_block = "\n".join(f"- {source}" for source in sources) # Make bullet list of sources
        assistant_text = f"{assistant_text}\n\nQuellen:\n{sources_block}" # Add bullet list after "Quellen:"

    return assistant_text # Return formatted text

In [8]:
def render_training_text(question, correct_answer, sources):
    # Build assistant text for question
    assistant_text = build_assistant_text(correct_answer, sources)

    # Format for training
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": str(question).strip()},
        {"role": "assistant", "content": assistant_text},
    ]

    # Apply and return chat template
    return tokenizer.apply_chat_template(
        messages,
        tokenize = False, # Return text, not ids
        add_generation_prompt = False, # Add no marker
    )

In [9]:
def build_text_records(records):
    text_records = [] # Empty list for records

    # Iterate through every record
    for record in records:
        # Create dictionairy with entries; keeps original fields but adds text field
        text_records.append({
            "id": str(record["id"]).strip(),
            "raw_question": str(record["prompt"]).strip(),
            "reference_answer": str(record["correct_answer"]).strip(),
            "reference_sources": normalize_sources(record["sources"]),
            "text": render_training_text(
                question = record["prompt"],
                correct_answer = record["correct_answer"],
                sources = record["sources"],
            ),
        })

    return text_records # Return record

In [10]:
def load_json(file_path):
  with open(file_path, "r", encoding = "utf-8") as f:
    raw_text = f.read() # Read contents of file

  try:
    parsed = json.loads(raw_text)

    # If dictionairy, convert it to a list
    if isinstance(parsed, dict):
      parsed = [parsed]

    if not isinstance(parsed, list):
      raise ValueError("The training JSON must be a JSON array")

  except:
    raise ValueError("Something is wrong with the JSON file")

  # Define required fields
  required_fields = {"id", "prompt", "correct_answer", "sources"}


  # Iterate over all entries, validate and normalize
  for idx, record in enumerate(parsed):
    if not isinstance(record, dict):
      raise ValueError(f"Record {idx} is not JSON")

    missing_fields = required_fields - set(record.keys()) # See if fields are missing

    if missing_fields:
      raise ValueError(f"Record {idx} is missing required fields: {missing_fields}")

  return build_text_records(parsed) # Return records of JSON entries

Now we will load and configure the model and tokenizer for further use. We start by getting the `model` and `tokenizer` objects.

In [11]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL_NAME, # Use base model
    max_seq_length = MAX_SEQ_LENGTH, # Set max. seq length
    load_in_4bit = LOAD_IN_4_BIT # Load in 4 bit for better performance
)

==((====))==  Unsloth 2026.4.4: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

Next, we will define the padding token to the EOS token.

In [12]:
if tokenizer.pad_token is None: # If no padding token is set
  tokenizer.pad_token = tokenizer.eos_token # Set padding token to EOS token

Now we will add the LoRA Adapters to the base model.

In [13]:
model = FastLanguageModel.get_peft_model(
    model, # Select model which was defined previously
    r = LORA_R, # Set LoRA rank
    target_modules = TARGET_MODULES, # Only specific sublayers receive low-rank updates
    lora_alpha = LORA_ALPHA, # Set alpha
    lora_dropout = LORA_DROPOUT, # Set dropout
    bias = "none", # Bias terms are not trained
    use_gradient_checkpointing = True, # Enable gradient checkpointing
    random_state = SEED # Set seed
)

Unsloth: Making `base_model.model.model.vision_tower.vision_model.embeddings` require gradients


Now we check the used device.

In [14]:
DEVICE = next(model.parameters()).device # Checking which device torch runs on
DEVICE

device(type='cuda', index=0)

Now we will continue the preparation. For this, we are first configuring the system prompt.

In [15]:
SYSTEM_PROMPT = """Du bist ein Assistent für österreichisches Steuerrecht.
Beantworte Fragen präzise, knapp und fachlich, wenn möglich in maximal 4-5 Sätzen, korrekt auf Deutsch.
Wenn die Rechtslage unklar ist oder dir Informationen fehlen, sage das offen.
Erfinde keine Gesetzesstellen oder Fakten. Zitiere verwendetete Paragraphen und Richtlinien des österreichen Rechts nach gültigen Zitierregeln.
"""

Now that we have defined the helper functions, we can use them to load the JSON data. We will then use the Dataset library to convert it into a HF dataset object.

In [16]:
ft_data = Dataset.from_list(load_json(SFT_DIR)) # Prepare fine-tuning dataset

For this data, we will perform a train-validation split.

In [17]:
ft_data = ft_data.train_test_split(
    test_size = VALIDATION_SPLIT, # Set split
    seed = SEED, # Set seed
    shuffle  = True # Shuffle before splitting (the dataset is ordered by topic, so shuffling is necessary)
)

Now we will extract the train and validation parts.

In [18]:
# Extract datasets
ft_train = ft_data["train"]
ft_eval = ft_data["test"]

Now we will define the training configuators using the previously defined parameters.

In [19]:
sft_config = SFTConfig(
  output_dir = CHECKPOINT_DIR, # Save checkpoints to path
  per_device_train_batch_size = TRAIN_BATCH_SIZE, # Set training batch size
  per_device_eval_batch_size = TRAIN_BATCH_SIZE, # Set evaluation batch size
  gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS, # Set gradient acc. steps
  num_train_epochs= NUM_EPOCHS, # Set number of epochs
  learning_rate = LEARNING_RATE, # Set learning rate
  logging_steps = LOGGING_STEPS, # Set logging steps
  eval_strategy = EVAL_STRATEGY, # Set eval strategy
  save_strategy = SAVE_STRATEGY, # Set save strategy
  save_total_limit = 2, # Prevent checkpoint accumulation
  load_best_model_at_end = True, # Load best model at the end (not really necessary as models will be reloaded)
  metric_for_best_model = "eval_loss", # Set eval_loss as metric
  greater_is_better = False, # Lower loss is better
  bf16 = USE_BF16, # Precisions setting
  fp16 = USE_FP16, # Precision setting
  report_to = "none", # Sent logs to no one
  seed = SEED, # Seed for reproduceability
  max_length = MAX_SEQ_LENGTH, # Token length cap during processing
  packing = False, # Do not fuse multiple samples into one long sequence
  dataset_text_field = "text" # Set field where inputs are
)

# Configure early stopping
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience = EARLY_STOPPING_PATIENCE,
    early_stopping_threshold = EARLY_STOPPING_THRESHOLD,
)

# Configure trainer
trainer = SFTTrainer(
    model = model, # Select model
    args = sft_config, # Handover arguments
    train_dataset = ft_train, # Select train dataset
    eval_dataset = ft_eval, # Select evaluation dataset
    processing_class = tokenizer,
    callbacks = [early_stopping_callback] # Set callback
)

print(trainer) # Print to see if everything worked

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/270 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/30 [00:00<?, ? examples/s]

Now that we have configured everything, we can start the actual fine-tuning.

In [20]:
train_result = trainer.train() # Perform actual training

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 270 | Num Epochs = 1 | Total steps = 34
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 32,788,480 of 4,332,867,952 (0.76% trained)


Epoch,Training Loss,Validation Loss
1,0.903140,0.858287


In [21]:
print(train_result) # Print training results

TrainOutput(global_step=34, training_loss=1.7225014321944292, metrics={'train_runtime': 1003.2158, 'train_samples_per_second': 0.269, 'train_steps_per_second': 0.034, 'total_flos': 1655959927964160.0, 'train_loss': 1.7225014321944292})


Now we will create the output directory for the adapter.

In [22]:
Path(ADAPTER_OUTPUT_DIR).mkdir(parents=True, exist_ok=True) # Create directory for LoRA adapter

Now we will same the model and the tokenizer to this created directory.

In [23]:
model.save_pretrained(ADAPTER_OUTPUT_DIR) # Save model

In [24]:
tokenizer.save_pretrained(ADAPTER_OUTPUT_DIR) # Save tokenizer

['/content/model2_lora_adapter/processor_config.json']

We wait for a few seconds until Colab has processed the files.

In [25]:
sleep(10)

We also want to dowload the LoRA-adapter.

In [26]:
zip_file = shutil.make_archive(
    str(ADAPTER_OUTPUT_DIR.parent / ADAPTER_OUTPUT_DIR.name),
    "zip",
    root_dir = str(ADAPTER_OUTPUT_DIR)
)

files.download(zip_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Task 2.2.2: Answering the questions from the dataset
Now that we have fine-tuned our model, we can use this model to answer the questions from the dataset.
For this, we are going to reuse logic from the first notebook.

First we will import the dataset as a Pandas dataframe.

In [27]:
df = pd.read_csv(DATASET_DIR) # Read dataset with the questions to answer

Now we will load the model and tokenizer again (so the fine-tuning does not have to be rerun everytime). For this, it is first checked if the path and the necessary files exists, then the model will be loaded.

In [28]:
adapter_path = Path(ADAPTER_OUTPUT_DIR) # Get adapter path

adapter_exists = (
    adapter_path.exists() # See if adapter path exists
    and (adapter_path / "adapter_config.json").exists() # See if config file exists
    and any(adapter_path.glob("adapter_model.*")) # See if model file exists
)

# If no adapter was found, fine-tuning has to be re-run or adapter has to be uploaded
if not adapter_exists:
  raise FileNotFoundError(
      f"No saved LoRA adapter was found in '{ADAPTER_OUTPUT_DIR}'. Run the fine-tuning in Section 2.2.1 first"
  )

In [29]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = str(adapter_path), # Set LoRA adapter as model, everything necessary will be collected
    max_seq_length = MAX_SEQ_LENGTH, # Maximal context length
    load_in_4bit = LOAD_IN_4_BIT, # Load model in 4 bit for better hardware performance
)

==((====))==  Unsloth 2026.4.4: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

In [30]:
FastLanguageModel.for_inference(model) # Switches to inference mode
model.eval() # Switches to evaluation mode

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForConditionalGeneration(
      (model): Gemma3Model(
        (vision_tower): SiglipVisionModel(
          (vision_model): SiglipVisionTransformer(
            (embeddings): SiglipVisionEmbeddings(
              (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
              (position_embedding): Embedding(4096, 1152)
            )
            (encoder): SiglipEncoder(
              (layers): ModuleList(
                (0-26): 27 x SiglipEncoderLayer(
                  (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                  (self_attn): SiglipAttention(
                    (k_proj): lora.Linear(
                      (base_layer): Linear(in_features=1152, out_features=1152, bias=True)
                      (lora_dropout): ModuleDict(
                        (default): Identity()
                      )
                      (lora_A): ModuleDict(
  

Now we will check if the model runs on the correct device.

In [31]:
DEVICE = next(model.parameters()).device # Checking which device the model runs on
DEVICE

device(type='cuda', index=0)

Now we will set the system prompt again (for the case that Section 2.2.1 has not been run completely).

In [32]:
SYSTEM_PROMPT = """Du bist ein Assistent für österreichisches Steuerrecht.
Beantworte Fragen präzise, knapp und fachlich, wenn möglich in maximal 4-5 Sätzen, korrekt auf Deutsch.
Wenn die Rechtslage unklar ist oder dir Informationen fehlen, sage das offen.
Erfinde keine Gesetzesstellen oder Fakten. Zitiere verwendetete Paragraphen und Richtlinien des österreichen Rechts nach gültigen Zitierregeln.
"""

Now, we will define a `build_prompt_question`-function that will create the prompt for a specific question.

In [33]:
def build_prompt_question(question: str): # Formats the prompts for use
    # As the model is multimodal, the type of the input has to be specified
    return [
        {"role": "system",
         "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",
         "content": [{"type": "text", "text": str(question).strip()}]}
    ]

Now we will define a function that will answer the questions in a batched manner.

In [34]:
def answer_questions_batched(questions, batch_size: int = INFER_BATCH_SIZE): # Function to answer the questions
    questions = list(questions) # Make sure "questions" is a list
    conversations = [build_prompt_question(q) for q in questions] # Build prompt for every question
    answers = [None] * len(conversations) # Create None-list with length of no. of questions

    # Apply padding
    ## Some tokenizers have the tokenizer in a additional attribute
    if hasattr(tokenizer, "tokenizer"): # If in additional attribute
        tokenizer.tokenizer.padding_side = "left" # Apply left padding
        if tokenizer.tokenizer.pad_token is None: # If Padding Token is None
            tokenizer.tokenizer.pad_token = tokenizer.tokenizer.eos_token # Set Padding Token to EOS token
        eos_id = tokenizer.tokenizer.eos_token_id # Set EOS ID
    else:
        tokenizer.padding_side = "left" # Apply left padding
        if tokenizer.pad_token is None: # If no padding token is set
            tokenizer.pad_token = tokenizer.eos_token # Set Padding token to EOS token
        eos_id = tokenizer.eos_token_id # Set EOS ID

    # This batches the question answering
    for batch_start in range(0, len(conversations), batch_size): # Range from 0 to n with batch_size spacing
        batch_convs = conversations[batch_start: batch_start + batch_size] # Get conversations for batch

        # Apply chat template to batch
        inputs = tokenizer.apply_chat_template(
            batch_convs, # The conversations
            tokenize = True, # Tokenizes the conversations
            add_generation_prompt = True, # Adds marker for completion
            return_dict = True, # Request dictionairy output; for accessing input_ids and attention_mask
            return_tensors = "pt", # Return PyTorch Tensors
            padding = True, # Apply padding (left as set before)
            truncation = True, # Cut off when too long
        ).to(DEVICE) # Send to device

        input_len = inputs["input_ids"].shape[1] # Get size of a answer (all same length due to padding)

        # Actually answer the questions
        ## Inference mode
        with torch.inference_mode():
            outputs = model.generate(
                **inputs, # Apply inputs from tokenizer
                max_new_tokens = MAX_NEW_TOKENS, # Maximum no. of new tokens
                do_sample = False, # Greedy decoing, better for reproduceability
                use_cache = True, # Use caching for better performance
                pad_token_id = eos_id, # Set Padding Token
                eos_token_id = eos_id, # Set EOS token
            )

        generated_only = outputs[:, input_len:] # Only keep generated answers
        decoded = tokenizer.batch_decode( # Decode answers
            generated_only,
            skip_special_tokens = True,
        )

        for i, answer in enumerate(decoded):# Go through decoded answers
            answers[batch_start + i] = answer.strip() # Clean answer and add to answers list with right index

    return answers # Return all answers

Now that we have defined to logic, we can combine the functions to start the generation of the answers.

In [35]:
df_prompts = df["prompt"].tolist() # Convert prompts-column to list

In [36]:
prompt_answers = answer_questions_batched(df_prompts, batch_size = INFER_BATCH_SIZE) # Generate answers in batches

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

Now we can add the generated answers back to the dataframe.

In [42]:
df["answer"] = prompt_answers # Add answers back to dataframe

The ID and answer columns can now be exported.

In [38]:
# Output only the desired columns ID and answer
df[["id", "answer"]].to_csv(OUTPUT_DIR, # to set output directory
                            index = False, # No index wanted
                            quotechar = '"') # Wrap cell text in quotes

Now we wait for a few seconds until Colab has processed the file.

In [39]:
sleep(10) # Wait for 10 seconds until Colab has processed the file

Now we can download the results.

In [40]:
files.download(OUTPUT_DIR.resolve()) # Automatically download file

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>